### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is used useful for the following: 

* Tracking agent behavior with logging, analytics and debugging. 
* Transforming prompts, tool selection, and output formatting. 
* Adding retries, fallbacks, and early termination logic.
* Applying root limits omar omar guardrails and PII Addiction

#### init

In [ ]:
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

os.environ["OPENAI_BASE_URL"] = "http://localhost:4000"
os.environ["OPENAI_API_KEY"] = "sk_dummy_key"

#### Summarization Middleware

Auto-summarizes conversation history when approaching rate limits, preserving recent messages while compressing old context

##### Message Count Trigger

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message-based summarization

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:openai/gpt-oss-120b",
            trigger=("messages", 10),
            keep=("messages", 4)
        )
    ]
)



In [ ]:
# First we need to create a thread, with its thread id
config={"configurable": {"thread_id":"test-1"}}

# Test Data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages": [HumanMessage(content=q)]}, config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")



##### Token Count Trigger

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="gpt-4o",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o",
            trigger=("tokens", 550),
            keep=("tokens", 200)
        )
    ]
)

In [ ]:
config = {"configurable": {"thread_id": "test_1"}}

# Token Counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token

In [ ]:
# Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config)

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(response["messages"])

##### Fraction-based Trigger

In [15]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool

@tool
def search_hotel(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}: Grand Hotel $350/night, City Inn $180/night, Budget Stay $75/night"""

agent = create_agent(
    model="gpt-4o",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o",
            trigger=("fraction", 0.005), # 0.5% = ~640 tokens
            keep=("fraction", 0.002) # 0.2% = ~256 tokens (I think this is specific to the context limit of the model used.)
        )
    ]
)

In [16]:
config = {"configurable": {"thread_id": "test_1"}}

# Token Counter (approximate)
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Run test

cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke({"messages": [HumanMessage(content=f"find hotels in {city}")]}, config=config)

    tokens = count_tokens(response["messages"])
    fraction = tokens / 1048576
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response["messages"])} messages")
    print(response["messages"])

Paris: ~55 tokens (0.0052%), 4 messages
[HumanMessage(content='find hotels in Paris', additional_kwargs={}, response_metadata={}, id='512b8915-4eab-4573-94ff-10650254c9eb'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 53, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': None, 'rejected_prediction_tokens': None, 'text_tokens': 16}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'text_tokens': 53}}, 'model_provider': 'openai', 'model_name': 'gpt-4o', 'system_fingerprint': None, 'id': '8v6EarPKBaWJ7M8PjOSRkAk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a01783-d894-79d2-90e0-5de984f47d1b-0', tool_calls=[{'name': 'search_hotel', 'args': {'city': 'Paris'}, 'id': 'call_76465__thought__El4KXAERTTIP/dfyLwS2im3D5VsIJQ/tKIHweRRO+